# Import Zone

In [ ]:
from qutip import *
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib
import sys
from pathlib import Path
import json
from lmfit import Model
from matplotlib.ticker import MaxNLocator
from itertools import accumulate



import scienceplots as splt
matplotlib.pyplot.style.use(['science', 'notebook', 'ieee'])
matplotlib.pyplot.rcParams['font.family'] = 'Times New Roman'


project_root = Path.cwd().parent.parent
sys.path.append(str(project_root / "src"))


from molecules.molecule import CaH, CaOH, CaOH_dm2, CaH_dm2
from QLS.state_dist import States
from BayesianSE.estimator import BayesianStateEstimation

from pumping.pump_evolution import run_pumping
from state_preparation import run_pumping_pipeline
from BayesianSE.main import run_bayesian_state_estimation
from BayesianSE._plotting import plot_method_comparison

from BayesianSE.estimator_measurements import meas_sensitivity_heatmap
from RAP.rap_utils import compute_transitions
from pumping.pump_utils import cut_trans_df
from saving import plot_final_state_pop
from pumping.pump_hamiltonians import H_bsb_manifold

# Off-resonant coupling

In [ ]:
def run_off_resonant_coupling(mo1, 
                              measurements, 
                              max_meas,
                              j_max,
                              rabi_by_j, 
                              keep_sub_manifold_levels, 
                              n_motional, 
                              index_motional, 
                              index_internal, 
                              heating_rate, 
                              time_steps_mol,
                              save_dir):
    
    for j_val_meas in range(0, max_meas):

        freq = measurements[j_val_meas][1]
        t_value = measurements[j_val_meas][2]

        is_minus = measurements[j_val_meas][5]

        for j_man in range(1, j_max + 1):

            print("Measurement j = ", j_val_meas+1)
            print("Measurement time = ", t_value)
            print("Measurement frequency = ", freq)
            print("Manifold j = ", j_man)
            print("initialization in ", index_internal, '\n')

            transitions_in_j = mo1.transition_df[mo1.transition_df["j"] == j_man]
        
            filtered_in_j = cut_trans_df(transitions_in_j, j_man, keep_sub_manifold_levels)

            sub_index = min(keep_sub_manifold_levels, 2*j_man + 1) 
            n_internal = 2*sub_index        
        
            assert 0 <= index_internal < n_internal, "index_internal out of range"

            psi_internal = basis(n_internal, index_internal)
            rho_internal = ket2dm(psi_internal)
        
            assert 0 <= index_motional < n_motional, "index_motional out of range"

            psi_motional = basis(n_motional, index_motional)
            rho_motional = ket2dm(psi_motional)

            rho0 = tensor(rho_internal, rho_motional)

            final_time = t_value*1.2   # us           
            times = np.linspace(0, final_time, time_steps_mol)

            laser_detuning = 2 * np.pi * freq       

            opts = Options(store_states=True, progress_bar="text", nsteps=20000)

            e_ops = []
            a = tensor(qeye(n_internal), destroy(n_motional))
            c_ops = [np.sqrt(heating_rate) * a.dag()]

            rabi_rate = next(
                rate for (j_min, j_max), rate in rabi_by_j.items() if j_min <= j_man <= j_max
            )

            H_tot, args = H_bsb_manifold(filtered_in_j, j_man, is_minus, n_motional, n_internal, rabi_rate, laser_detuning)

            result_rr1 = mesolve(H_tot, rho0, times, c_ops, e_ops, args=args, options=opts)

            # plot_internal_and_motional_dynamics(result_rr1, times, n_internal, n_motional, j_val_meas, t_value=t_value)

            full_path = plot_final_state_pop(result_rr1, mo1, index_internal, j_val_meas+1, j_man, is_minus, n_internal, n_motional, rabi_rate, save_dir, t_value=t_value)

## CaH Molecule creation

In [ ]:
b_field_gauss = 3.6
j_max = 14

mo1 = CaH.create_molecule_data(b_field_gauss=b_field_gauss, j_max=j_max)

temperature = 300
states1 = States(mo1, temperature)

dephased = False
coherence_time_us = 1600
is_minus = False
noise_params = None
seed = None
max_excitation = 0.9

Estimator = BayesianStateEstimation(model=mo1, 
                                    temperature=temperature, 
                                    b_field_gauss=b_field_gauss, 
                                    j_max=j_max,
                                    false_negative_rate=0,
                                    false_positive_rate=0.)

## Initialization in the leftmost state

In [ ]:
index_internal = 0
max_meas = 1

keep_sub_manifold_levels = 5
n_motional = 4
index_motional = 0
heating_rate = 1.0 / 1e6
time_steps_mol = 200
save_dir = "cah_init_LM"

### Constant Rabi rates

In [ ]:
rabi_by_j = {
    # (1, 10): 2 * np.pi * 0.0001,     
    (1, 14): 2 * np.pi * 0.002,    
    # (37, 50): 2 * np.pi * 0.005     
}

In [ ]:
import warnings
warnings.filterwarnings("ignore")


Estimator.measurement_setting(rabi_by_j=rabi_by_j, 
                              dephased=dephased, 
                              coherence_time_us=coherence_time_us, 
                              is_minus=is_minus, 
                              noise_params=noise_params, 
                              seed=seed, 
                              max_excitation=max_excitation,
                              laser_miscalibration=None,
                              seed_miscalibration=None,
                              marginalization=False)

measurements = Estimator.measurements


run_off_resonant_coupling(mo1, 
                          measurements, 
                          max_meas,
                          j_max,
                          rabi_by_j, 
                          keep_sub_manifold_levels, 
                          n_motional, 
                          index_motional, 
                          index_internal, 
                          heating_rate, 
                          time_steps_mol,
                          save_dir)

## Initialization in the penultimate upper state

In [ ]:
index_internal = 1
max_meas = 14


keep_sub_manifold_levels = 5
n_motional = 4
index_motional = 0
heating_rate = 1.0 / 1e6
time_steps_mol = 200
save_dir = "cah_init_PU"

In [ ]:
rabi_by_j = {
    # (1, 10): 2 * np.pi * 0.0001,     
    (1, 14): 2 * np.pi * 0.002,    
    # (37, 50): 2 * np.pi * 0.005     
}

In [ ]:
import warnings
warnings.filterwarnings("ignore")


Estimator.measurement_setting(rabi_by_j=rabi_by_j, 
                              dephased=dephased, 
                              coherence_time_us=coherence_time_us, 
                              is_minus=is_minus, 
                              noise_params=noise_params, 
                              seed=seed, 
                              max_excitation=max_excitation,
                              laser_miscalibration=None,
                              seed_miscalibration=None,
                              marginalization=False)

measurements = Estimator.measurements


run_off_resonant_coupling(mo1, 
                          measurements, 
                          max_meas,
                          j_max,
                          rabi_by_j, 
                          keep_sub_manifold_levels, 
                          n_motional, 
                          index_motional, 
                          index_internal, 
                          heating_rate, 
                          time_steps_mol,
                          save_dir)